In [ ]:
db = FSDB(environ.get('ROMI_DB', "/data/ROMI/Romi_Alexis/analyse_jo"))
db.connect(unsafe=True)

In [ ]:
fileset_names = locate_task_filesets(scan, ["images", "Colmap", "Mask"])

In [ ]:
mask_files = mask_fs.get_files()

In [ ]:
bbox = colmap_fs.get_metadata('bounding_box')

In [ ]:
x_min, x_max = 300, 435
y_min, y_max = 300, 435
z_min, z_max = -300, 60

In [ ]:
# - Defines the origin of the voxel array:
origin = np.array([x_min, y_min, z_min])

In [ ]:
for mask_file in mask_files:
    mask = o3d.geometry.Image(read_image(mask_file))
    mask_md = mask_file.get_metadata()
    cam_md = mask_md["colmap_camera"]['camera_model']
    intrinsic_md = get_camera_kwargs_from_images_metadata(mask_file)
    if intrinsic_md['model'].lower() != "opencv":
        opencv_cam = colmap_params_from_kwargs(**intrinsic_md)
        intrinsic_md = dict(zip(['fx', 'fy', 'cx', 'cy', 'k1', 'k2', 'p1', 'p2'], opencv_cam))
    cam_intrinsic = o3d.camera.PinholeCameraIntrinsic()
    cam_intrinsic.set_intrinsics(width=cam_md['width'], height=cam_md['height'],
                                 fx=intrinsic_md['fx'], fy=intrinsic_md['fy'],
                                 cx=intrinsic_md['cx'], cy=intrinsic_md['cy'])
    cam_params = o3d.camera.PinholeCameraParameters()
    cam_params.intrinsic = cam_intrinsic
    cam_extrinsic = np.array([[0., 0., 0., 0.], [0., 0., 0., 0.], [0., 0., 0., 0.], [0., 0., 0., 1.]])
    rotmat = np.array(mask_md["colmap_camera"]["rotmat"])
    tvec = np.array(mask_md["colmap_camera"]["tvec"])
    cam_extrinsic[:3, :3] = rotmat.T
    cam_extrinsic[:3, 3] = -rotmat.T @ tvec
    cam_params.extrinsic = np.linalg.inv(cam_extrinsic)
    vx_grid.carve_silhouette(mask, cam_params)

In [ ]:
-Rt@t

In [ ]:
cam_params.intrinsic.intrinsic_matrix

In [ ]:
mask = o3d.geometry.Image(read_image(mask_files[0]))